# ✈️ FlightRadar24 - ETL & Airflow Pipeline ( Python CRONJOB)

## Objectif

Ce projet a pour but de construire un pipeline **ETL industrialisé**, tolérant aux erreurs et observable, qui récupère les données de vol en temps réel depuis l’API FlightRadar24 toutes les **2 heures**, les nettoie, les transforme, puis les stocke sous **format Parquet** (ou CSV).  
Ces données sont ensuite analysées via **PySpark** pour générer des **indicateurs métier** sur le trafic aérien mondial.

### Installation des dépendances pour le projet

In [ ]:
!pip install FlightRadarAPI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 29.1 MB/s eta 0:00:00


In [ ]:
!pip install findspark

## Extarct data from flight radar api

In [ ]:
####################
#                  #
# Authored BY : me #
#                  #
####################

# Extract data from flightRadar api sous format dataframe

import pandas as pd
import logging
from datetime import datetime, timezone
import os
from FlightRadar24 import FlightRadar24API

# Setup des logs
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def extract_flights():
    api = FlightRadar24API()
    flights = api.get_flights()

    logger.info(f"{len(flights)} vols récupérés")

    data = []
    for flight in flights:
        try:
            data.append({
                "id": flight.id,
                "callsign": flight.callsign,
                "airline_iata": flight.airline_iata,
                "airline_icao": flight.airline_icao,
                "origin": flight.origin_airport_iata,
                "destination": flight.destination_airport_iata,
                "aircraft_code": flight.aircraft_code,
                "registration": flight.registration,
                "latitude": flight.latitude,
                "longitude": flight.longitude,
                "altitude": flight.altitude,
                "ground_speed": flight.ground_speed,
                "heading": flight.heading,
                "vertical_speed": flight.vertical_speed,
                "on_ground": flight.on_ground,
                "squawk": flight.squawk,
                "time": flight.time
            })
        except Exception as e:
            logger.warning(f"Vol ignoré à cause d'une erreur : {e}")

    return pd.DataFrame(data)

In [ ]:
df = extract_flights()

In [ ]:
# Display  head 5
df.head(5)

,id,callsign,airline_iata,airline_icao,origin,destination,aircraft_code,registration,latitude,longitude,altitude,ground_speed,heading,vertical_speed,on_ground,squawk,time
0,3a3e4634,HBAL732,,,,,BALL,N257TH,37.9563,-104.2741,58600,8,150,-64,0,,1752402167
1,3a9f267b,RPT1,,,,,GLID,RPT-1,-33.3772,-70.5786,2503,1,52,320,0,,1752402114
2,3a9f26a5,LILC,,,,,GRND,,45.8080,8.7723,800,0,180,0,0,,1752402136
3,3a9f26d1,EGC194,,,,,GLID,EGC19-4,36.9170,-4.6831,1354,2,23,0,0,,1752402163
4,3b0eacaf,HBAL756,,,,,BALL,N256TH,43.6216,-97.1159,57600,5,135,0,0,,1752402167


--> let's do some data exploration to understand our data, so that we can do the data cleaning

### EDA( exploration data analysis )

In [ ]:
print("Columns \n",df.columns)

Columns 
 Index(['id', 'callsign', 'airline_iata', 'airline_icao', 'origin',
       'destination', 'aircraft_code', 'registration', 'latitude', 'longitude',
       'altitude', 'ground_speed', 'heading', 'vertical_speed', 'on_ground',
       'squawk', 'time'],
      dtype='object')


In [ ]:
print("types \n", df.dtypes)

types 
 id                 object
callsign           object
airline_iata       object
airline_icao       object
origin             object
destination        object
aircraft_code      object
registration       object
latitude          float64
longitude         float64
altitude            int64
ground_speed        int64
heading             int64
vertical_speed      int64
on_ground           int64
squawk             object
time                int64
dtype: object


In [ ]:
# shape
df.shape

(1500, 17)

In [ ]:
# Valeurs uniques
print("Valeurs unique \n",df.nunique().sort_values(ascending=False))

Valeurs unique 
 id                1500
longitude         1498
latitude          1497
callsign          1484
registration      1468
ground_speed       376
altitude           369
heading            333
destination        256
origin             253
airline_icao       190
airline_iata       183
aircraft_code       88
vertical_speed      65
time                57
on_ground            2
squawk               1
dtype: int64


In [ ]:
# check for Duplicates for id

print("duplicates \n",df.duplicated(subset=['id']).sum())

duplicates 
 0


In [ ]:
# Statistiques descriptives

print("Statistiques descriptives \n", df.describe(include='all'))

Statistiques descriptives 
               id callsign airline_iata airline_icao origin destination  \
count       1500     1500         1500         1500   1500        1500   
unique      1500     1484          183          190    253         256   
top     3b3afc3b     GLF6                                                
freq           1        5          125          121     93         124   
mean         NaN      NaN          NaN          NaN    NaN         NaN   
std          NaN      NaN          NaN          NaN    NaN         NaN   
min          NaN      NaN          NaN          NaN    NaN         NaN   
25%          NaN      NaN          NaN          NaN    NaN         NaN   
50%          NaN      NaN          NaN          NaN    NaN         NaN   
75%          NaN      NaN          NaN          NaN    NaN         NaN   
max          NaN      NaN          NaN          NaN    NaN         NaN   

       aircraft_code registration     latitude    longitude       altitude  \
count

--> IL faut commentez

In [ ]:
## Répartition des compagnies

In [ ]:
print("Répartition des compagnies   \n", df["airline_icao"].value_counts().head(10))

Répartition des compagnies   
 airline_icao
       121
UAE     84
QTR     83
AAL     79
UAL     77
DAL     50
THY     47
BAW     31
AFR     29
JBU     28
Name: count, dtype: int64


In [ ]:
# Répartition des origines :

In [ ]:
df["destination"].value_counts().head(10)

print("Répartition des origines   \n", df["origin"].value_counts().head(10))

Répartition des origines   
 origin
       93
DXB    74
DOH    63
LAX    62
JFK    59
SFO    46
ICN    41
PVG    34
SYD    33
ORD    33
Name: count, dtype: int64


In [ ]:
# Répartition des destinations :

print("Répartition des destinations   \n", df["destination"].value_counts().head(10))

Répartition des destinations   
 destination
       124
LHR     87
CDG     51
JFK     42
FRA     37
MAD     36
HKG     33
AMS     29
LAX     28
IST     27
Name: count, dtype: int64


### Plots

--> To sum Up about our Data :

    # callsign, airline_iata, airline_icao : pour identifier les compagnies

    # aircraft_code, registration, squawk : pour identifier les avions

    # latitude, longitude, altitude : position

    # origin, destination : codes IATA

    # ground_speed, heading, vertical_speed, on_ground : pour l'état du vol

    # time : timestamp (en secondes POSIX, à convertir si besoin)

## Transform Data

### Data Cleaning

In [ ]:
# Valeurs manquantes
missing_values = (df.isna().sum() + (df == "").sum()).sort_values(ascending=False)
print("Valeurs manquantes (en nombre) \n", missing_values)

Valeurs manquantes (en nombre) 
 squawk            1500
airline_iata       125
destination        124
airline_icao       121
origin              93
registration        33
callsign             5
id                   0
aircraft_code        0
latitude             0
longitude            0
ground_speed         0
altitude             0
heading              0
vertical_speed       0
on_ground            0
time                 0
dtype: int64


In [ ]:
# Pourcentage de valeurs manquantes, correctement affiché en %
missing_percent = ((df.isna().sum() + (df == "").sum()) / len(df) * 100).sort_values(ascending=False)
print("Valeurs manquantes en % :\n", missing_percent.round(2))

Valeurs manquantes en % :
 squawk            100.00
airline_iata        8.33
destination         8.27
airline_icao        8.07
origin              6.20
registration        2.20
callsign            0.33
id                  0.00
aircraft_code       0.00
latitude            0.00
longitude           0.00
ground_speed        0.00
altitude            0.00
heading             0.00
vertical_speed      0.00
on_ground           0.00
time                0.00
dtype: float64


In [ ]:
import pandas as pd
import logging

logger = logging.getLogger(__name__)

def clean_flights_data(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("Nettoyage des données de vol...")

    # Étape 1 : Supprimer les colonnes avec + de 50% de valeurs manquantes
    missing_ratio = (df.isna().sum() + (df == "").sum()) / len(df)
    cols_to_drop = missing_ratio[missing_ratio > 0.5].index.tolist()
    if cols_to_drop:
        logger.info(f"Colonnes supprimées (plus de 50% de valeurs manquantes) : {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    # Étape 2 : Supprimer les lignes avec au moins une valeur manquante ou vide
    initial_row_count = len(df)
    df = df[~(df.isna() | (df == "")).any(axis=1)]
    removed_rows = initial_row_count - len(df)
    logger.info(f"Lignes supprimées pour valeurs manquantes : {removed_rows}")

    logger.info("Nettoyage terminé.")
    return df


In [ ]:
df_cleaned = clean_flights_data(df)

In [ ]:
# Pourcentage de valeurs manquantes, correctement affiché en %
missing_percent = ((df_cleaned.isna().sum() + (df_cleaned == "").sum()) / len(df_cleaned) * 100).sort_values(ascending=False)
print("Valeurs manquantes en % après le cleaning :\n", missing_percent.round(2))

Valeurs manquantes en % après le cleaning :
 id                0.0
callsign          0.0
airline_iata      0.0
airline_icao      0.0
origin            0.0
destination       0.0
aircraft_code     0.0
registration      0.0
latitude          0.0
longitude         0.0
altitude          0.0
ground_speed      0.0
heading           0.0
vertical_speed    0.0
on_ground         0.0
time              0.0
dtype: float64


--> No missing data

## Load Data into CSV format ( Or parquet )

In [ ]:
import os
from datetime import datetime, timezone
import pandas as pd

def save_to_csv(df: pd.DataFrame, base_path="Flights/rawzone") -> str:
    """
    Sauvegarde le DataFrame au format CSV avec une nomenclature horodatée.
    Exemple de chemin : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713124500.csv
    """
    now = datetime.now(timezone.utc)

    # Création du chemin horodaté
    path = os.path.join(
        base_path,
        f"tech_year={now.year}",
        f"tech_month={now.strftime('%Y-%m')}",
        f"tech_day={now.strftime('%Y-%m-%d')}"
    )
    os.makedirs(path, exist_ok=True)

    # Nom du fichier CSV
    filename = f"flights_{now.strftime('%Y%m%d%H%M%S')}.csv"
    full_path = os.path.join(path, filename)

    # Sauvegarde en CSV
    df.to_csv(full_path, index=False)
    print(f"[INFO] Fichier CSV sauvegardé : {full_path}")

    return full_path


In [ ]:
df_cleaned = clean_flights_data(df)
save_to_csv(df_cleaned)

[INFO] Fichier CSV sauvegardé : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713114308.csv


'Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713114308.csv'

## Spark Analysis


In [ ]:
import os
import glob
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, row_number, desc
from pyspark.sql.window import Window

def run_spark_analysis():
    print("🚀 Début de l’analyse Spark")

    spark = SparkSession.builder.appName("FlightAnalysis").getOrCreate()

    # Chercher le dernier fichier CSV/Parquet dans la rawzone
    files = glob.glob("Flights/rawzone/tech_year=*/tech_month=*/tech_day=*/flights_*.csv")
    if not files:
        raise FileNotFoundError("Aucun fichier trouvé dans la rawzone.")

    latest_file = sorted(files)[-1]
    print(f" Chargement du fichier : {latest_file}")
    df = spark.read.csv(latest_file, header=True, inferSchema=True)

    # Nettoyage minimal
    clean_df = df.filter(
        (col("latitude").isNotNull()) &
        (col("longitude").isNotNull()) &
        (col("airline_iata").isNotNull()) &
        (col("on_ground") == 0)
    )

    # -------------------------------
    # 1. Compagnie avec le plus de vols en cours
    # -------------------------------
    print("\n 1. Compagnie avec le plus de vols en cours :")
    top_airline = (
        clean_df.filter((col("airline_iata").isNotNull()) & (col("airline_iata") != ""))
        .groupBy("airline_iata")
        .count()
        .orderBy(desc("count"))
        .first()
    )
    if top_airline:
        print(f"✈️ {top_airline['airline_iata']} avec {top_airline['count']} vols.")
    else:
        print(" Données insuffisantes.")


    # -------------------------------
    # 2. Compagnie avec le plus de vols régionaux par continent
    # -------------------------------
    print("\n 2. Compagnie avec le plus de vols régionaux par continent :")
    if "origin" in clean_df.columns and "destination" in clean_df.columns:
        regional_flights = clean_df.filter(
            (col("origin").isNotNull()) &
            (col("destination").isNotNull()) &
            (col("origin") == col("destination")) &
            (col("airline_iata").isNotNull()) & (col("airline_iata") != "")
        )
        window = Window.partitionBy("origin").orderBy(desc("count"))
        top_regional = (
            regional_flights.groupBy("origin", "airline_iata")
            .count()
            .withColumn("rank", row_number().over(window))
            .filter(col("rank") == 1)
            .collect()
        )
        for row in top_regional:
            print(f" {row['origin']} ➤ {row['airline_iata']} ({row['count']} vols)")
    else:
        print("⚠️ Colonnes 'origin' ou 'destination' absentes.")

    # -------------------------------
    # 3. Vol avec le trajet le plus long
    # -------------------------------
    print("\n 3. Vol en cours avec le trajet le plus long :")

    # distance_km difference entre destination et origin
    if "distance_km" in clean_df.columns:
        longest_flight = (
            clean_df.filter(col("distance_km").isNotNull())
            .orderBy(desc("distance_km"))
            .select("callsign", "airline_iata", "distance_km")
            .first()
        )
        if longest_flight:
            print(f" {longest_flight['callsign']} ({longest_flight['airline_iata']}) : {round(longest_flight['distance_km'], 2)} km")
    elif "altitude" in clean_df.columns:
        longest_flight = (
            clean_df.filter(col("altitude").isNotNull())
            .orderBy(desc("altitude"))
            .select("callsign", "airline_iata", "altitude")
            .first()
        )
        if longest_flight:
            print(f" {longest_flight['callsign']} ({longest_flight['airline_iata']}) : {longest_flight['altitude']} pieds")
    else:
        print("⚠️ Aucune donnée de distance ni d'altitude.")


    # -------------------------------
    # 4 . Pour chaque continent, la longueur de vol moyenne
    # -------------------------------
    iata_to_continent = {
        "AF": "Europe",     # Air France
        "LH": "Europe",     # Lufthansa
        "DL": "North America",  # Delta
        "EK": "Asia",       # Emirates (Moyen-Orient = Asie)
        "BA": "Europe",     # British Airways
        "NH": "Asia",       # All Nippon Airways
        "QF": "Oceania",    # Qantas
        # Ajoute ce dont tu as besoin
    }

    continent_df = spark.createDataFrame(
        [(k, v) for k, v in iata_to_continent.items()],
        ["airline_iata", "continent"]
    )

    flights_with_continent = clean_df.join(continent_df, on="airline_iata", how="left")

    avg_altitude = (
        flights_with_continent
        .filter(col("altitude").isNotNull() & col("continent").isNotNull())
        .groupBy("continent")
        .agg(avg("altitude").alias("avg_altitude"))
        .orderBy("continent")
    )

    # Affichage clair
    print("\n 4. Altitude moyenne des vols en cours par continent (approximation de la longueur) :")
    for row in avg_altitude.collect():
        print(f" {row['continent']} ➤ {round(row['avg_altitude'], 2)} pieds")


    # -------------------------------
    # 5 . Constructeur avec le plus de vols actifs
    # -------------------------------

    print("\n 5. Constructeur d’avions avec le plus de vols actifs :")
    if "aircraft_code" in clean_df.columns:
        top_manufacturer = (
            clean_df.filter((col("aircraft_code").isNotNull()) & (col("aircraft_code") != ""))
            .groupBy("aircraft_code")
            .count()
            .orderBy(desc("count"))
            .first()
        )
        if top_manufacturer:
            print(f" {top_manufacturer['aircraft_code']} ({top_manufacturer['count']} vols)")
    else:
        print("⚠️ Colonne 'aircraft_code' absente.")


    # -------------------------------
    # 6. Top 3 modèles par pays de la compagnie
    # -------------------------------

    # Il faut faire un mapping des noms de compagnie pour ajouter une nouvelle colonne country

    iata_to_country = {
        "AF": "France",
        "LH": "Germany",
        "DL": "USA",
        "EK": "UAE",
        "BA": "UK",
        # ajoute d'autres si besoin...
    }

    mapping_df = spark.createDataFrame(
        [(k, v) for k, v in iata_to_country.items()],
        ["airline_iata", "country"]
    )

    enriched_df = clean_df.join(mapping_df, on="airline_iata", how="left")

    from pyspark.sql.window import Window

    print("\n 6. Top 3 modèles d’avion par pays de la compagnie :")

    # On garde que les lignes valides
    valid_flights = enriched_df.filter(
        col("country").isNotNull() & col("aircraft_code").isNotNull()
    )

    # Regrouper et compter
    top_models = valid_flights.groupBy("country", "aircraft_code").count()

    # Appliquer le classement
    window = Window.partitionBy("country").orderBy(desc("count"))
    ranked_models = top_models.withColumn("rank", row_number().over(window)) \
                              .filter(col("rank") <= 3) \
                              .orderBy("country", "rank")

    # Afficher
    grouped = ranked_models.collect()
    from collections import defaultdict
    result = defaultdict(list)
    for row in grouped:
        result[row["country"]].append((row["aircraft_code"], row["count"]))

    for country, models in result.items():
        models_str = ", ".join([f"{m} ({c} vols)" for m, c in models])
        print(f"🌍 {country} ➤ {models_str}")



## Pipeline ETL

In [ ]:
import logging
from datetime import datetime
import traceback

def run_pipeline():
    logger = logging.getLogger("FlightRadarETL")
    logging.basicConfig(level=logging.INFO)

    try:
        logger.info("🛫 Lancement du pipeline ETL FlightRadar24")

        # 1. Extraction
        from FlightRadar24 import FlightRadar24API
        fr_api = FlightRadar24API()
        flights = fr_api.get_flights()

        logger.info(f"Nombre de vols extraits : {len(flights)}")

        import pandas as pd
        df = pd.DataFrame([flight.__dict__ for flight in flights])

        # 2. Nettoyage
        # from data_cleaning import clean_flights_data  # ou remplace par ta fonction locale
        df_cleaned = clean_flights_data(df)

        # 3. Sauvegarde (CSV ou Parquet)
        # from file_saver import save_to_csv  # ou ta fonction locale
        path_saved = save_to_csv(df_cleaned)

        logger.info(f" Données sauvegardées dans : {path_saved}")

        # 4. Déclenchement de l’analyse Spark
        run_spark_analysis()

    except Exception as e:
        logger.error(f" Une erreur est survenue : {e}")
        traceback.print_exc()


### Tester de la pipeline

In [ ]:
run_pipeline()

[INFO] Fichier CSV sauvegardé : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713122005.csv
🚀 Début de l’analyse Spark
 Chargement du fichier : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713122005.csv

 1. Compagnie avec le plus de vols en cours :
✈️ QR avec 83 vols.

 2. Compagnie avec le plus de vols régionaux par continent :
⚠️ Colonnes 'origin' ou 'destination' absentes.

 3. Vol en cours avec le trajet le plus long :
 VJT793 (VJ) : 47000 pieds

 4. Altitude moyenne des vols en cours par continent (approximation de la longueur) :
 Asia ➤ 34053.42 pieds
 Europe ➤ 34705.43 pieds
 North America ➤ 33980.42 pieds
 Oceania ➤ 32061.61 pieds

 5. Constructeur d’avions avec le plus de vols actifs :
 B77W (189 vols)

 6. Top 3 modèles d’avion par pays de la compagnie :
🌍 France ➤ A359 (9 vols), B77W (6 vols), B789 (2 vols)
🌍 Germany ➤ A359 (7 vols), B77L (5 vols), A343 (3 vols)
🌍 UAE ➤ B77W (34 vols), A388 (24 vols)

## Lancement du Cronjonb à fréquence de 2 h

In [ ]:

import logging
from datetime import datetime
import traceback
# from my_etl.etl_pipeline import run_pipeline  ###

def main():
    logging.basicConfig(
        filename="pipeline.log",  # fichier de logs
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )

    logger = logging.getLogger("FlightRadarETL")
    logger.info("🔁 Lancement automatique du pipeline...")

    try:
        run_pipeline()
        logger.info("✅ Pipeline terminé avec succès.")
    except Exception as e:
        logger.error(f" Erreur dans le pipeline : {e}")
        logger.error(traceback.format_exc())

if __name__ == "__main__":
    main()


[INFO] Fichier CSV sauvegardé : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713122027.csv
🚀 Début de l’analyse Spark
 Chargement du fichier : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713122027.csv

 1. Compagnie avec le plus de vols en cours :
✈️ QR avec 82 vols.

 2. Compagnie avec le plus de vols régionaux par continent :
⚠️ Colonnes 'origin' ou 'destination' absentes.

 3. Vol en cours avec le trajet le plus long :
 VJT793 (VJ) : 47000 pieds

 4. Altitude moyenne des vols en cours par continent (approximation de la longueur) :
 Asia ➤ 34019.86 pieds
 Europe ➤ 34629.09 pieds
 North America ➤ 33893.68 pieds
 Oceania ➤ 32017.17 pieds

 5. Constructeur d’avions avec le plus de vols actifs :
 B77W (189 vols)

 6. Top 3 modèles d’avion par pays de la compagnie :
🌍 France ➤ A359 (9 vols), B77W (6 vols), B789 (2 vols)
🌍 Germany ➤ A359 (7 vols), B77L (5 vols), A343 (3 vols)
🌍 UAE ➤ B77W (34 vols), A388 (24 vols)

In [ ]:
import time
import logging
import traceback
from datetime import datetime

log_file = "pipeline.log"
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("FlightRadarETL")

def run_pipeline_job():
    logger.info(" Lancement du job ETL")
    print("\n[⏳] Lancement du job ETL...")

    try:
        run_pipeline()
        logger.info("Pipeline terminé avec succès.")
        print("[✅] Pipeline terminé avec succès.")
    except Exception as e:
        logger.error(f" Erreur dans le pipeline : {e}")
        logger.error(traceback.format_exc())
        print(f"[❌] Erreur dans le pipeline : {e}")

# Boucle de test : exécuter toutes les 2 minutes
for i in range(3):
    print(f"\n=========== 🔄 Lancement #{i+1} à {datetime.now().strftime('%H:%M:%S')} ===========")
    run_pipeline_job()
    print("\n ")
    print(f"[🕒] Prochain lancement dans 2 minutes...\n")
    time.sleep(2 * 60)



=========== 🔄 Lancement #1 à 12:25:49 ===========

[⏳] Lancement du job ETL...
[INFO] Fichier CSV sauvegardé : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713122549.csv
🚀 Début de l’analyse Spark
 Chargement du fichier : Flights/rawzone/tech_year=2025/tech_month=2025-07/tech_day=2025-07-13/flights_20250713122549.csv

 1. Compagnie avec le plus de vols en cours :
✈️ QR avec 79 vols.

 2. Compagnie avec le plus de vols régionaux par continent :
⚠️ Colonnes 'origin' ou 'destination' absentes.

 3. Vol en cours avec le trajet le plus long :
 VJT793 (VJ) : 47000 pieds

 4. Altitude moyenne des vols en cours par continent (approximation de la longueur) :
 Asia ➤ 33560.3 pieds
 Europe ➤ 34145.76 pieds
 North America ➤ 35019.71 pieds
 Oceania ➤ 35364.88 pieds

 5. Constructeur d’avions avec le plus de vols actifs :
 B77W (191 vols)

 6. Top 3 modèles d’avion par pays de la compagnie :
🌍 France ➤ A359 (10 vols), B77W (7 vols), B789 (2 vols)
🌍 Germany ➤ A3

KeyboardInterrupt: 

### Lancement d'un Job chaque 2 min ( `while True:` pour exécution infinie )

In [ ]:
import time

while True:
    try:
        run_pipeline()
        print(" Pipeline exécutée avec succès.")
    except Exception as e:
        print(f" Erreur : {e}")
    print("\n ")
    print("⏳ En attente de 2h...")
    print("\n ")
    time.sleep(2 * 60 * 60)
